### Build drivers dimension
1. Read "silver" drivers table
2. Read "gold" ref_nationality_region table
3. Join data from drivers with ref_nationality_region using nationality
4. Select required columns
    -   drivers.driver_id
    -   drivers.driver_name
    -   drivers.date_of_birth
    -   drivers.nationality
    -   ref_nationality_region.region
7. Write transformed data to gold dim_drivers table

In [0]:
%run ../Workspace/common/configuration_environment

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

##### 1. Read source table
    a)   drivers
    b)   ref_nationality_region

In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers")
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [0]:
display(drivers_df)
display(ref_nationality_region_df)

date_of_birth,driver_id,name,nationality,ingestion_timestamp,source_file,driver_name
1940-04-19,ahrens,"List(kurt, ahrens)",German,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Kurt Ahrens
1961-04-20,barilla,"List(paolo, barilla)",Italian,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Paolo Barilla
1914-02-28,bayol,"List(élie, bayol)",French,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Élie Bayol
1924-01-07,birger,"List(pablo, birger)",Argentine,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Pablo Birger
1946-11-25,borgudd,"List(slim, borgudd)",Swedish,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Slim Borgudd
1916-09-15,branca,"List(toni, branca)",Swiss,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Toni Branca
1949-12-24,brown,"List(warwick, brown)",Australian,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Warwick Brown
1924-04-04,christie,"List(bob, christie)",American,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Bob Christie
1936-03-04,clark,"List(jim, clark)",British,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,Jim Clark
1906-08-16,george_connor,"List(george, connor)",American,2026-04-27T13:15:15.144381Z,dbfs:/Volumes/formula1/landing/files/drivers.json,George Connor


nationality,region
Puerto Rican,North America
Haitian,North America
Jamaican,North America
Puerto Rican,North America
Chinese,Asia
Japanese,Asia
Korean,Asia
Indian,Asia
Indonesian,Asia
Malaysian,Asia


3. Join data from drivers with ref_nationality_region using nationality
4. Select required columns
    -   drivers.driver_id
    -   drivers.driver_name
    -   drivers.date_of_birth
    -   drivers.nationality
    -   ref_nationality_region.region

In [0]:
dim_drivers_df = (
    drivers_df
    .join(
        ref_nationality_region_df,
        drivers_df.nationality == ref_nationality_region_df.nationality,
        'left'
    )
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias('nationality_region')
    )
)

In [0]:
display(dim_drivers_df)

driver_id,driver_name,date_of_birth,nationality,nationality_region
ahrens,Kurt Ahrens,1940-04-19,German,Europe
barilla,Paolo Barilla,1961-04-20,Italian,Europe
bayol,Élie Bayol,1914-02-28,French,Europe
birger,Pablo Birger,1924-01-07,Argentine,South America
borgudd,Slim Borgudd,1946-11-25,Swedish,Europe
branca,Toni Branca,1916-09-15,Swiss,Europe
brown,Warwick Brown,1949-12-24,Australian,Oceania
christie,Bob Christie,1924-04-04,American,North America
clark,Jim Clark,1936-03-04,British,Europe
george_connor,George Connor,1906-08-16,American,North America


In [0]:
(
    dim_drivers_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)    
)

In [0]:
display (dim_drivers_df)

driver_id,driver_name,date_of_birth,nationality,nationality_region
ahrens,Kurt Ahrens,1940-04-19,German,Europe
barilla,Paolo Barilla,1961-04-20,Italian,Europe
bayol,Élie Bayol,1914-02-28,French,Europe
birger,Pablo Birger,1924-01-07,Argentine,South America
borgudd,Slim Borgudd,1946-11-25,Swedish,Europe
branca,Toni Branca,1916-09-15,Swiss,Europe
brown,Warwick Brown,1949-12-24,Australian,Oceania
christie,Bob Christie,1924-04-04,American,North America
clark,Jim Clark,1936-03-04,British,Europe
george_connor,George Connor,1906-08-16,American,North America
